In [71]:
import pandas as pd
import torch
from pylate import models, retrieve, indexes
from tqdm.auto import tqdm

In [72]:
# Load the modernColbert
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = models.ColBERT(model_name_or_path="lightonai/colbertv2.0", device=device)

In [73]:
# Load the existing Voyager index
try:
    index = indexes.Voyager(
        index_folder="../data/processed/pylate_index",
        index_name="esci_data_index",
        override=False
    )
except RuntimeError as e:
    print(f"Warning: Could not load existing index ({e}).\n"
          "The index file is corrupted or missing. Creating a new empty index.\n"
          "Run the indexing notebook (indexing_data.ipynb) on Colab to rebuild it.")
    index = indexes.Voyager(
        index_folder="../data/processed/pylate_index",
        index_name="esci_data_index",
        override=True
    )
index.ef_search = 128
retriever = retrieve.ColBERT(index=index)

In [74]:
import subprocess, time, requests

try:
    requests.get("http://localhost:11434", timeout=2)
    print("Ollama already running")
except Exception:
    subprocess.Popen(["ollama", "serve"])
    time.sleep(5)
    print("Ollama started")

subprocess.run(["ollama", "pull", "llama3.2"], check=True)

OLLAMA_URL = "http://localhost:11434/api/chat"

def expand_query(query):
    data = {
        "model": "llama3.2",
        "stream": False,
        "messages": [
            {
                "role": "system",
                "content": "You are a search engine optimizer. Provide 5 synonyms or related product terms for the user query. Output ONLY the terms separated by commas. No explanations."
            },
            {
                "role": "user",
                "content": f"Query: {query}"
            }
        ]
    }
    response = requests.post(OLLAMA_URL, json=data)
    response.raise_for_status()
    content = response.json()["message"]["content"]
    return [t.strip() for t in content.split(",")]


Ollama already running


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [75]:
import sqlite3
from sqlitedict import SqliteDict

# Load the Amazon ESCI dataset
test_df = pd.read_parquet("../data/raw/shopping_queries_dataset_examples.parquet",
                         columns=['query_id', 'query', 'product_id', 'esci_label'])

# Get the pids ACTUALLY in the Voyager index via its own metadata
doc_map = SqliteDict(
    "../data/processed/pylate_index/esci_data_index/document_ids_to_embeddings.sqlite",
    outer_stack=False
)
indexed_pids = set(int(k) for k in doc_map.keys())
doc_map.close()

# Look up their ASINs from SQLite
conn = sqlite3.connect("../data/processed/products_dataset.db")
all_products = pd.read_sql("SELECT pid, product_id FROM products", conn)
conn.close()

indexed_asins = set(
    all_products[all_products['pid'].isin(indexed_pids)]['product_id']
)

print(f"Voyager token vectors:   {index.index.num_elements}")
print(f"Actual indexed docs:     {len(indexed_pids)}")
print(f"Matching ASINs:          {len(indexed_asins)}")
print(f"Total products in DB:    {len(all_products)}")
print(f"Total test rows:         {len(test_df)}")

# Filter ground truth to only products the retriever can actually return
test_df_filtered = test_df[test_df['product_id'].isin(indexed_asins)]

# Only keep queries that have at least one annotated product in the index
valid_query_ids = test_df_filtered['query_id'].unique()
print(f"Queries with indexed ground truth: {len(valid_query_ids)}")

if len(valid_query_ids) == 0:
    raise RuntimeError(
        "No matching queries found — the index is empty or was built from different products.\n"
        "Rebuild the index by running indexing_data.ipynb on Google Colab, "
        "then download pylate_index.tar.gz to data/processed/ and extract it."
    )

# Sample up to 100 unique queries from the filtered set
n_sample = min(100, len(valid_query_ids))
eval_queries = (
    test_df_filtered
    .drop_duplicates('query_id')
    .sample(n_sample, random_state=42)
)
print(f"Sampled eval queries:    {len(eval_queries)}")


Voyager token vectors:   3218213
Actual indexed docs:     50560
Matching ASINs:          50560
Total products in DB:    405284
Total test rows:         2621288
Queries with indexed ground truth: 5501
Sampled eval queries:    100


In [76]:
eval_queries['esci_label'].unique()

<ArrowStringArray>
['E', 'C', 'S', 'I']
Length: 4, dtype: str

In [77]:
eval_queries['esci_label'].value_counts()

esci_label
E    69
S    20
I     7
C     4
Name: count, dtype: int64

In [78]:
# Test a single query before running the whole loop
try:
    expanded = expand_query("test")
    all_queries = ["test"] + expanded
    test_embs = model.encode(all_queries, is_query=True, show_progress_bar=False)
    test_res = retriever.retrieve(test_embs, k=10)
    print("Search with query expansion is working!")
    print(f"Expanded terms: {expanded}")
except Exception as e:
    print(f" Search still failing: {e}")


Retrieving documents (bs=50): 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]

Search with query expansion is working!
Expanded terms: ['test', 'examination', 'assessment', 'evaluation', 'inspection']


In [79]:
# Retrieval and label extraction (with query expansion)
conn = sqlite3.connect("../data/processed/products_dataset.db")
cursor = conn.cursor()

# Create a mapping for the ESCI shorthand codes
CODE_TO_NAME = {
    "E": "exact",
    "S": "substitute",
    "C": "complement",
    "I": "irrelevant"
}

K = 5
evaluation_results = []

for _, row in tqdm(eval_queries.iterrows(), total=len(eval_queries)):
    q_id = row['query_id']
    q_text = row['query']

    # Expand query with Ollama and encode all variants
    all_queries = [q_text] + expand_query(q_text)
    query_embs = model.encode(all_queries, is_query=True, show_progress_bar=False)
    raw_hits = retriever.retrieve(query_embs, k=K)

    # Merge: keep best score per pid across all expanded queries
    merged = {}
    for query_hits in raw_hits:
        for hit in query_hits:
            pid = hit['id']
            if pid not in merged or hit['score'] > merged[pid]:
                merged[pid] = hit['score']
    retrieved_pids = [
        pid for pid, _ in sorted(merged.items(), key=lambda x: x[1], reverse=True)[:K]
    ]

    # Map integer pid → ASIN + title via SQLite
    placeholders = ",".join("?" * len(retrieved_pids))
    cursor.execute(
        f"SELECT pid, product_id, product_title FROM products WHERE pid IN ({placeholders})",
        retrieved_pids
    )
    pid_rows = {r[0]: (r[1], r[2]) for r in cursor.fetchall()}  # pid -> (asin, title)

    # Ground truth labels for this query (only indexed products)
    query_ground_truth = test_df_filtered[test_df_filtered['query_id'] == q_id]

    hits_with_labels = []
    for rank, pid in enumerate(retrieved_pids):
        asin, title = pid_rows.get(pid, (None, "Title Not Found"))
        if asin is not None:
            match = query_ground_truth[query_ground_truth['product_id'] == asin]
            if not match.empty:
                raw_code = match.iloc[0]['esci_label']
                label = CODE_TO_NAME.get(raw_code, "irrelevant")
            else:
                label = "irrelevant"
        else:
            label = "irrelevant"

        hits_with_labels.append({
            "rank": rank + 1,
            "product_id": asin,
            "title": title,
            "label": label
        })

    evaluation_results.append({
        "query": q_text,
        "hits": hits_with_labels
    })

conn.close()


  0%|          | 0/100 [00:00<?, ?it/s]

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]


In [80]:
# Example output for the first query
first_query = evaluation_results[1]
print(f"Query: {first_query['query']}")
for hit in first_query['hits']:
    print(f"Rank {hit['rank']}: PID {hit['product_id']} Product: {hit['title']}: -> Label: {hit['label']}")


Query: 28 palms linen
Rank 1: PID B076JBBBMT Product: 28 Palms Men's Relaxed-Fit 9" Inseam Linen Short with Drawstring, Tan, X-Large: -> Label: exact
Rank 2: PID B076J2PPNV Product: 28 Palms Men's Relaxed-Fit 11" Inseam Linen Short with Drawstring, Tan, X-Large: -> Label: exact
Rank 3: PID B076J3BCL3 Product: 28 Palms Men's Relaxed-Fit Long-Sleeve 100% Linen Shirt, White, X-Large: -> Label: exact
Rank 4: PID B07HMLFT65 Product: 28 Palms Women's 100% Linen Tropical Print Sleeveless Shift Dress Fronds White/Blue, Large: -> Label: exact
Rank 5: PID B07FL5G568 Product: Amazon Brand - 28 Palms Men's Relaxed-Fit Linen Pant with Drawstring, Natural, Large/30 Inseam: -> Label: exact


In [81]:
import numpy as np

# Relevance scores for hits (stored as full words by the retrieval loop)
RELEVANCE_MAP = {
    "exact": 1.0,
    "substitute": 0.1,
    "complement": 0.01,
    "irrelevant": 0.0
}

# Relevance scores for raw ESCI codes (used in test_df_filtered ground truth)
ESCI_CODE_MAP = {
    "E": 1.0,
    "S": 0.1,
    "C": 0.01,
    "I": 0.0
}

def get_ndcg(hits, query_id, k=5):
    actual_relevance = [RELEVANCE_MAP.get(h['label'].lower(), 0.0) for h in hits[:k]]
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(actual_relevance))

    # IDCG: use raw ESCI codes from test_df_filtered
    query_gt = test_df_filtered[test_df_filtered['query_id'] == query_id]
    ideal_relevance = sorted(
        [ESCI_CODE_MAP.get(l, 0.0) for l in query_gt['esci_label']],
        reverse=True
    )
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevance[:k]))

    return dcg / idcg if idcg > 0 else 0.0


In [82]:
# Aggregate nDCG@K scores across all sampled queries
K = 5
all_ndcg_scores = []

for result, (_, eval_row) in zip(evaluation_results, eval_queries.iterrows()):
    q_id = eval_row['query_id']
    score = get_ndcg(result['hits'], q_id, k=K)
    all_ndcg_scores.append(score)

mean_ndcg = np.mean(all_ndcg_scores)
print(f"--- Final Evaluation Results ---")
print(f"Mean nDCG@{K}: {mean_ndcg:.4f}")


--- Final Evaluation Results ---
Mean nDCG@5: 0.4945


In [83]:
def judge_relevance(query, product_title):
    """Ask llama3.2 to rate relevance using the same ESCI label scheme."""
    data = {
        "model": "llama3.2",
        "stream": False,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a product search relevance judge.\n"
                    "Rate the relevance of a product to a search query using exactly one label:\n"
                    "E - Exact: product directly satisfies the query\n"
                    "S - Substitute: product could substitute for what was searched\n"
                    "C - Complement: product complements but does not match the query\n"
                    "I - Irrelevant: product is not relevant to the query\n"
                    "Output ONLY a single letter: E, S, C, or I"
                )
            },
            {
                "role": "user",
                "content": f"Query: {query}\nProduct: {product_title}"
            }
        ]
    }
    response = requests.post(OLLAMA_URL, json=data)
    response.raise_for_status()
    text = response.json()["message"]["content"].strip().upper()
    for char in text:
        if char in "ESCI":
            return char
    return "I"  # default if response can't be parsed


In [84]:
# Judge relevance for all retrieved hits using llama3.2
# Reuses evaluation_results from the query expansion run above (same K=5 config)
llm_judged_results = []

for result, (_, eval_row) in tqdm(
    zip(evaluation_results, eval_queries.iterrows()),
    total=len(evaluation_results),
    desc="LLM judging"
):
    llm_hits = []
    for hit in result['hits']:
        title = hit['title']
        if title and title != "Title Not Found":
            llm_code = judge_relevance(result['query'], title)
        else:
            llm_code = "I"
        llm_hits.append({
            **hit,
            "llm_label": llm_code,
            "llm_score": ESCI_CODE_MAP[llm_code],
        })
    llm_judged_results.append({
        "query":    result['query'],
        "query_id": eval_row['query_id'],
        "hits":     llm_hits,
    })


LLM judging:   0%|          | 0/100 [00:00<?, ?it/s]

In [86]:
# nDCG using LLM-assigned scores for DCG; ESCI ground truth for IDCG
def get_llm_ndcg(llm_hits, query_id, k=5):
    actual = [h['llm_score'] for h in llm_hits[:k]]
    dcg    = sum(rel / np.log2(i + 2) for i, rel in enumerate(actual))
    query_gt = test_df_filtered[test_df_filtered['query_id'] == query_id]
    ideal    = sorted([ESCI_CODE_MAP.get(l, 0.0) for l in query_gt['esci_label']], reverse=True)
    idcg     = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal[:k]))
    return dcg / idcg if idcg > 0 else 0.0

llm_ndcg_scores = [
    get_llm_ndcg(r['hits'], r['query_id']) for r in llm_judged_results
]
esci_ndcg_scores = [
    get_ndcg(r['hits'], eval_row['query_id'])
    for r, (_, eval_row) in zip(evaluation_results, eval_queries.iterrows())
]

print("--- LLM-as-Judge vs ESCI Ground Truth ---")
print(f"nDCG@5 (ESCI ground truth): {np.mean(esci_ndcg_scores):.4f}")
print(f"nDCG@5 (LLM judge):         {np.mean(llm_ndcg_scores):.4f}")

# Label agreement between LLM and ESCI ground truth
CODE_TO_NAME = {"E": "exact", "S": "substitute", "C": "complement", "I": "irrelevant"}
agreed, total = 0, 0
for result in llm_judged_results:
    for hit in result['hits']:
        esci_label = hit['label']           # full word from retrieval loop
        llm_label  = CODE_TO_NAME.get(hit['llm_label'], 'irrelevant')
        if esci_label == llm_label:
            agreed += 1
        total += 1

print(f"\nLabel agreement (LLM vs ESCI): {agreed}/{total} = {agreed/total:.1%}")

# Per-label breakdown
from collections import Counter
esci_counts = Counter()
llm_counts  = Counter()
for result in llm_judged_results:
    for hit in result['hits']:
        esci_counts[hit['label']] += 1
        llm_counts[CODE_TO_NAME.get(hit['llm_label'], 'irrelevant')] += 1

print("\nLabel distribution (ESCI vs LLM):")
print(f"{'Label':<12} {'ESCI':>6} {'LLM':>6}")
for label in ['exact', 'substitute', 'complement', 'irrelevant']:
    print(f"{label:<12} {esci_counts[label]:>6} {llm_counts[label]:>6}")


--- LLM-as-Judge vs ESCI Ground Truth ---
nDCG@5 (ESCI ground truth): 0.4945
nDCG@5 (LLM judge):         0.4880

Label agreement (LLM vs ESCI): 222/500 = 44.4%

Label distribution (ESCI vs LLM):
Label          ESCI    LLM
exact           198    161
substitute       15    197
complement        2     12
irrelevant      285    130


In [87]:
def run_evaluation(eval_queries, test_df_filtered, model, retriever,
                   cached_expansions, expand_n=5, k_multiplier=1,
                   original_weight=1.0, K=5):
    CODE_TO_NAME = {"E": "exact", "S": "substitute", "C": "complement", "I": "irrelevant"}
    conn = sqlite3.connect("../data/processed/products_dataset.db")
    cursor = conn.cursor()
    results = []

    for _, row in tqdm(eval_queries.iterrows(), total=len(eval_queries), leave=False):
        q_id   = row['query_id']
        q_text = row['query']

        expanded   = cached_expansions.get(q_text, [])[:expand_n]
        all_queries = [q_text] + expanded

        query_embs = model.encode(all_queries, is_query=True, show_progress_bar=False)
        fetch_k    = K * k_multiplier
        raw_hits   = retriever.retrieve(query_embs, k=fetch_k)

        merged = {}
        for q_idx, query_hits in enumerate(raw_hits):
            weight = original_weight if q_idx == 0 else 1.0
            for hit in query_hits:
                pid     = hit['id']
                weighted = hit['score'] * weight
                if pid not in merged or weighted > merged[pid]:
                    merged[pid] = weighted

        retrieved_pids = [
            pid for pid, _ in sorted(merged.items(), key=lambda x: x[1], reverse=True)[:K]
        ]

        placeholders = ",".join("?" * len(retrieved_pids))
        cursor.execute(
            f"SELECT pid, product_id, product_title FROM products WHERE pid IN ({placeholders})",
            retrieved_pids
        )
        pid_rows = {r[0]: (r[1], r[2]) for r in cursor.fetchall()}

        query_gt = test_df_filtered[test_df_filtered['query_id'] == q_id]
        hits_with_labels = []
        for rank, pid in enumerate(retrieved_pids):
            asin, title = pid_rows.get(pid, (None, "Title Not Found"))
            if asin is not None:
                match = query_gt[query_gt['product_id'] == asin]
                label = CODE_TO_NAME.get(match.iloc[0]['esci_label'], "irrelevant") if not match.empty else "irrelevant"
            else:
                label = "irrelevant"
            hits_with_labels.append({"rank": rank + 1, "product_id": asin, "title": title, "label": label})

        results.append({"query": q_text, "query_id": q_id, "hits": hits_with_labels})

    conn.close()
    return results


In [88]:
# Pre-compute Ollama expansions once — reused across all configs
print("Caching query expansions (this calls Ollama once per unique query)...")
cached_expansions = {}
for q_text in tqdm(eval_queries['query'].unique(), desc="Expanding"):
    cached_expansions[q_text] = expand_query(q_text)
print(f"Cached {len(cached_expansions)} queries.")
sample_q = eval_queries['query'].iloc[0]
print(f"\nExample — '{sample_q}' → {cached_expansions[sample_q]}")


Caching query expansions (this calls Ollama once per unique query)...


Expanding:   0%|          | 0/100 [00:00<?, ?it/s]

Cached 100 queries.

Example — 'arrowhead spring water' → ['Natural spring water', 'bottled mineral water', 'alkaline spring water', 'purified well water', 'electrolyte enhanced water']


In [89]:
def mean_ndcg(results, k=5):
    scores = [get_ndcg(r['hits'], r['query_id'], k) for r in results]
    return float(np.mean(scores))

BASELINE_SCORE = 0.6148  # no expansion (from earlier run)
EXPANSION_SCORE = float(np.mean([
    get_ndcg(r['hits'], eval_row['query_id'])
    for r, (_, eval_row) in zip(evaluation_results, eval_queries.iterrows())
]))  # current 5-term expansion

configs = [
    {"label": "5 terms, no weighting (current)",   "expand_n": 5, "k_multiplier": 1, "original_weight": 1.0},
    {"label": "M1 – 2 terms",                      "expand_n": 2, "k_multiplier": 1, "original_weight": 1.0},
    {"label": "M1 – 3 terms",                      "expand_n": 3, "k_multiplier": 1, "original_weight": 1.0},
    {"label": "M2 – wider fetch (k×2)",             "expand_n": 5, "k_multiplier": 2, "original_weight": 1.0},
    {"label": "M3 – boost original (×1.5)",         "expand_n": 5, "k_multiplier": 1, "original_weight": 1.5},
    {"label": "M3 – boost original (×2.0)",         "expand_n": 5, "k_multiplier": 1, "original_weight": 2.0},
    {"label": "M1+M3 – 2 terms + boost ×1.5",       "expand_n": 2, "k_multiplier": 1, "original_weight": 1.5},
    {"label": "M2+M3 – wider fetch + boost ×1.5",   "expand_n": 5, "k_multiplier": 2, "original_weight": 1.5},
]

comparison = [
    {"config": "No expansion (baseline)", "nDCG@5": BASELINE_SCORE, "vs_baseline": "—"},
]

for cfg in configs:
    label = cfg.pop("label")
    print(f"Running: {label} ...", end=" ", flush=True)
    res   = run_evaluation(eval_queries, test_df_filtered, model, retriever,
                           cached_expansions, K=5, **cfg)
    score = mean_ndcg(res)
    delta = score - BASELINE_SCORE
    comparison.append({"config": label, "nDCG@5": score, "vs_baseline": f"{delta:+.4f}"})
    print(f"nDCG@5 = {score:.4f}  ({delta:+.4f} vs baseline)")

print("\n" + "─" * 62)
print(f"{'Configuration':<38} {'nDCG@5':>7}  {'vs baseline':>10}")
print("─" * 62)
best = max(comparison, key=lambda x: x['nDCG@5'])['nDCG@5']
for row in comparison:
    marker = " ◀" if row['nDCG@5'] == best else ""
    print(f"{row['config']:<38} {row['nDCG@5']:>7.4f}  {row['vs_baseline']:>10}{marker}")
print("─" * 62)


Running: 5 terms, no weighting (current) ... 

  0%|          | 0/100 [00:00<?, ?it/s]

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]

nDCG@5 = 0.4358  (-0.1790 vs baseline)
Running: M1 – 2 terms ... 

  0%|          | 0/100 [00:00<?, ?it/s]

...

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00,  3.83it/s]

nDCG@5 = 0.6134  (-0.0014 vs baseline)
Running: M2+M3 – wider fetch + boost ×1.5 ... 

  0%|          | 0/100 [00:00<?, ?it/s]

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]

nDCG@5 = 0.6130  (-0.0018 vs baseline)

──────────────────────────────────────────────────────────────
Configuration                           nDCG@5  vs baseline
──────────────────────────────────────────────────────────────
No expansion (baseline)                 0.6148           — ◀
5 terms, no weighting (current)         0.4358     -0.1790
M1 – 2 terms                            0.4937     -0.1211
M1 – 3 terms                            0.4514     -0.1634
M2 – wider fetch (k×2)                  0.4358     -0.1790
M3 – boost original (×1.5)              0.6130     -0.0018
M3 – boost original (×2.0)              0.6148     -0.0000
M1+M3 – 2 terms + boost ×1.5            0.6134     -0.0014
M2+M3 – wider fetch + boost ×1.5        0.6130     -0.0018
──────────────────────────────────────────────────────────────


In [90]:
# M1+M3 – 2 expansion terms + original query boost ×2.0
res_2t_2x = run_evaluation(
    eval_queries, test_df_filtered, model, retriever,
    cached_expansions, expand_n=2, k_multiplier=1, original_weight=2.0, K=5
)
score_2t_2x = mean_ndcg(res_2t_2x)
print(f"2 terms + boost ×2.0:  nDCG@5 = {score_2t_2x:.4f}  ({score_2t_2x - BASELINE_SCORE:+.4f} vs baseline)")


  0%|          | 0/100 [00:00<?, ?it/s]

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00,  2.99it/s]

2 terms + boost ×2.0:  nDCG@5 = 0.6148  (-0.0000 vs baseline)
